# Customer revenue pipeline — segmented model

Train one small model per acquisition channel, then score the shared test set.

In [ ]:
# @node: Train segmented model  [transform]  in=train_df<-preprocessing:Train test split.train_df,test_df<-preprocessing:Train test split.test_df,feature_cols<-preprocessing:Train test split.feature_cols  out=segmented_model,segmented_scores,segmented_predictions
import numpy as np
import pandas as pd

segment_features = [c for c in feature_cols if not c.startswith("channel_")]

def fit_linear(df, columns):
    x = np.column_stack([np.ones(len(df)), df[columns].to_numpy(dtype=float)])
    y = df["revenue"].to_numpy(dtype=float)
    return np.linalg.lstsq(x, y, rcond=None)[0]

def predict_linear(df, columns, coef):
    x = np.column_stack([np.ones(len(df)), df[columns].to_numpy(dtype=float)])
    return x @ coef

global_coef = fit_linear(train_df, segment_features)
channel_models = {}
for channel, group in train_df.groupby("channel"):
    channel_models[channel] = fit_linear(group, segment_features) if len(group) >= 30 else global_coef

pred_parts = []
for channel, group in test_df.groupby("channel"):
    coef = channel_models.get(channel, global_coef)
    scored = group[["date", "channel", "region", "revenue"]].copy()
    scored["prediction"] = predict_linear(group, segment_features, coef).round(2)
    pred_parts.append(scored)

segmented_predictions = pd.concat(pred_parts).sort_index()
segmented_predictions["model"] = "segmented by channel"
actual = segmented_predictions["revenue"].to_numpy(dtype=float)
pred = segmented_predictions["prediction"].to_numpy(dtype=float)
err = pred - actual
segmented_scores = pd.DataFrame(
    [
        {
            "model": "segmented by channel",
            "rmse": float(np.sqrt(np.mean(err**2))),
            "mae": float(np.mean(np.abs(err))),
            "bias": float(np.mean(err)),
        }
    ]
)
segmented_model = {
    "kind": "channel_segmented_linear",
    "features": segment_features,
    "channels": sorted(channel_models),
}
display(segmented_scores.round(2))